In [24]:
import pandas as pd
import os
import glob
from pathlib import Path
import statsmodels.api as sm
import numpy as np

# Alpha

In [25]:
df_rf1 = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\Alpha\CMMPI_Omod.csv")
df_rf2 = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\Alpha\MBK_SHIBORM.csv")

In [26]:
df_rf1 = df_rf1[df_rf1['Omod05'] == '3m(91d)'].copy()
df_rf1 = df_rf1[df_rf1['Omod02'] == '价格'].copy()
df_rf2 = df_rf2[df_rf2['Term'] == '90天'].copy()
df_rf1 = df_rf1.copy()
df_rf1['Datesgn'] = pd.to_datetime(df_rf1['Datesgn'])  # 确保是 datetime
df_rf1['month'] = df_rf1['Datesgn'].dt.to_period('M')  # 取年月（如 2025-01）
monthly_avg1 = (df_rf1.groupby('month', as_index=False)['Omod07']
      .mean()
      .rename(columns={'Omod07': 'rate'}))
df_rf2 = df_rf2.copy()
df_rf2['SgnDate'] = pd.to_datetime(df_rf2['SgnDate'])  # 确保是 datetime
df_rf2['month'] = df_rf2['SgnDate'].dt.to_period('M')  # 取年月（如 2025-01）
monthly_avg2 = (df_rf2.groupby('month', as_index=False)['Shibor']
      .mean()
      .rename(columns={'Shibor': 'rate'}))
df_rf = pd.concat([monthly_avg1,monthly_avg2])
df_rf.rename(columns={'month': 'TradingMonth'}, inplace=True)
df_rf['rate'] = df_rf['rate'] / 100.0
df_rf['rate'] = (1 + df_rf['rate']) ** (1/12) -1

In [27]:
df_f = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\Alpha\STK_MKT_CARHARTFOURFACTORS.csv")

In [28]:
df_f = df_f[df_f['MarkettypeID'] == 'P9706'].copy()
df_f.drop(columns = 'MarkettypeID',inplace = True)

In [29]:
df_fr1 = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\Alpha\Fund_NAV_Month.csv")
df_fr2 = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\Alpha\Fund_NAV_Month1.csv")
df_fr = pd.concat([df_fr1,df_fr2])
df_temp = pd.read_csv(r"D:\studystudy\Window Dressing\merged data\merged_DependentVariable.csv")
df_s2m = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\Alpha\FUND_FundCodeInfo.csv")

In [30]:
# 1) Symbol -> MasterFundCode 映射（保留 df_fr 的所有列，并新增 MasterFundCode）
map_s2m = df_s2m[['Symbol', 'MasterFundCode']].drop_duplicates(subset=['Symbol'])
df_fr_m = df_fr.merge(map_s2m, on='Symbol', how='left')
# 2) 用 df_temp 的 MasterFundCode 过滤（只保留能在因变量表里对上的主基金）
keep_master = set(df_temp['MasterFundCode'].dropna().unique())
df_fr_m = df_fr_m[df_fr_m['MasterFundCode'].isin(keep_master)].copy()

In [31]:
df_alpha_reg_y = df_fr_m[['TradingMonth','MasterFundCode','ReturnAccumulativeNAV']].copy()
df_alpha_reg_y['TradingMonth'] = pd.to_datetime(df_alpha_reg_y['TradingMonth'])  
df_f['TradingMonth'] = pd.to_datetime(df_f['TradingMonth']) 
df_alpha_reg_y['TradingMonth'] = df_alpha_reg_y['TradingMonth'].dt.to_period('M')  
df_f['TradingMonth'] = df_f['TradingMonth'].dt.to_period('M')  
df_alpha_reg_y = df_alpha_reg_y.merge(df_rf , on = 'TradingMonth',how = 'left')
df_alpha_reg_y['Ri-Rf'] = df_alpha_reg_y['ReturnAccumulativeNAV'] - df_alpha_reg_y['rate']
df_reg_temp = df_alpha_reg_y[['TradingMonth','MasterFundCode','Ri-Rf']].merge(df_f , on = 'TradingMonth' , how = 'left')
df_reg_temp.dropna(inplace = True)

In [32]:
FACTOR_COLS = ['RiskPremium1', 'SMB1', 'HML1', 'UMD1']
Y_COL = 'Ri-Rf'
WINDOW = 24
df = df_reg_temp.copy()
# 1) TradingMonth：兼容 Period / datetime / string，统一为 Period[M]
if pd.api.types.is_period_dtype(df['TradingMonth']):
    df['TradingMonth'] = df['TradingMonth'].dt.asfreq('M')
else:
    df['TradingMonth'] = pd.to_datetime(df['TradingMonth'], errors='coerce').dt.to_period('M')
# 2) 确保数值列为 numeric
num_cols = [Y_COL] + FACTOR_COLS
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
# 3) 去掉关键列缺失，并排序、去重（防止同一基金同一月多条）
df = df.dropna(subset=['TradingMonth', 'MasterFundCode'] + num_cols).copy()
df = df.sort_values(['MasterFundCode', 'TradingMonth'])
df = df.drop_duplicates(subset=['MasterFundCode', 'TradingMonth'], keep='last')
def rolling_residual_alpha(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values('TradingMonth').copy()
    y = g[Y_COL].to_numpy()
    X = g[FACTOR_COLS].to_numpy()
    alpha_t = np.full(len(g), np.nan)
    # 用过去24个月（含当月）估计系数，并用当月残差作为 alpha_month
    for t in range(WINDOW - 1, len(g)):
        y_win = y[t - WINDOW + 1: t + 1]
        X_win = X[t - WINDOW + 1: t + 1]
        X_win_c = sm.add_constant(X_win, has_constant='add')
        model = sm.OLS(y_win, X_win_c).fit()
        y_hat_t = model.predict(sm.add_constant(X[[t]], has_constant='add'))[0]
        alpha_t[t] = y[t] - y_hat_t  # 当月异常收益（残差）
    out = g[['TradingMonth', 'MasterFundCode']].copy()
    out['alpha_month'] = alpha_t
    return out
# ===== 第一步产出：每月异常收益 =====
df_alpha_month = (df.groupby('MasterFundCode', group_keys=False)
      .apply(rolling_residual_alpha)
      .reset_index(drop=True))
# ===== 第二步产出：半年 Alpha（6个月加总） =====
df_alpha_month['Year'] = df_alpha_month['TradingMonth'].dt.year
df_alpha_month['Month'] = df_alpha_month['TradingMonth'].dt.month
df_alpha_month['Half'] = np.where(df_alpha_month['Month'] <= 6, 1, 2)
df_alpha_half = (df_alpha_month.dropna(subset=['alpha_month'])
                  .groupby(['MasterFundCode', 'Year', 'Half'], as_index=False)['alpha_month']
                  .sum()
                  .rename(columns={'alpha_month': 'Alpha_6m'}))
# 你现在会得到：
# df_alpha_month: TradingMonth, MasterFundCode, MarkettypeID, alpha_month
# df_alpha_half : MasterFundCode, MarkettypeID, Year, Half, Alpha_6m

C:\Users\MLTZ\AppData\Local\Temp\ipykernel_22492\3772106143.py:6: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_period_dtype(df['TradingMonth']):
C:\Users\MLTZ\AppData\Local\Temp\ipykernel_22492\3772106143.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(rolling_residual_alpha)


# Turnover

In [33]:
df_tur = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\others\RESSET_FDSTKTRDCOSTINC_1.csv", encoding="gb18030")

In [34]:
df_turn_std = df_tur.copy()
# 1) 统一基金代码列并映射 MasterFundCode
df_turn_std['Symbol'] = pd.to_numeric(df_turn_std['基金代码_FdCd'], errors='coerce').astype('Int64')
df_turn_std = df_turn_std.merge(
    map_s2m[['Symbol', 'MasterFundCode']],
    on='Symbol',
    how='left'
)
# 2) 处理报告日期，生成 Year / Half
df_turn_std['RepDt'] = pd.to_datetime(df_turn_std['报告日期_RepDt'], errors='coerce')
df_turn_std['TurningRate'] = pd.to_numeric(df_turn_std['近半年股票换手率(%)_TurnRt'], errors='coerce')
df_turn_std = df_turn_std.dropna(subset=['MasterFundCode', 'RepDt', 'TurningRate']).copy()
df_turn_std['Year'] = df_turn_std['RepDt'].dt.year.astype(int)
df_turn_std['Half'] = np.where(df_turn_std['RepDt'].dt.month <= 6, 1, 2).astype(int)
# 3) 标准化：只保留主键 + TurningRate，并在同一主基金同一期内取均值去重
df_turn_std = (
    df_turn_std.groupby(['MasterFundCode', 'Year', 'Half'], as_index=False)['TurningRate']
               .mean()
)
df_turn_std['MasterFundCode'] = df_turn_std['MasterFundCode'].astype(int)

# others

In [35]:
df_ta = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\others\FUND_FIN_Balance.csv")
df_fee = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\others\FUND_FIN_Income.csv")
df_inst = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\others\Fund_HolderStructure.csv")
df_inst['InstitutionRatio'] = df_inst['InstitutionRatio'].fillna(0)
df_fee = df_fee[df_fee['DATA_TYPE_ID'] == 3].copy()

In [36]:
df_ta["EndDate"] = pd.to_datetime(df_ta["EndDate"], errors="coerce")
df_ta["Year"] = df_ta["EndDate"].dt.year
df_ta["Half"] = df_ta["ReportTypeID"].map({5: 1, 6: 2}).astype("Int64")
df_ta = (
    df_ta.sort_values("EndDate")
         .groupby(["MasterFundCode", "Year", "Half"], as_index=False)
         .tail(1)[["MasterFundCode", "Year", "Half", "TotalAsset"]]
         .reset_index(drop=True))
df_ta = df_ta[['MasterFundCode','Year','Half','TotalAsset']].copy()
df_fee["EndDate"] = pd.to_datetime(df_fee["EndDate"], errors="coerce")
df_fee["Year"] = df_fee["EndDate"].dt.year
df_fee["Half"] = df_fee["ReportTypeID"].map({5: 1, 6: 2}).astype("Int64")
df_fee = (
    df_fee.groupby(["MasterFundCode", "Year", "Half"], as_index=False)["TotalOperatingCost"].sum())
# 把年报累计费用改成“下半年当期费用”：H2 = FY - H1
h1 = (
    df_fee[df_fee["Half"] == 1][["MasterFundCode", "Year", "TotalOperatingCost"]]
    .rename(columns={"TotalOperatingCost": "TotalOperatingCost_H1"})
)
df_fee = df_fee.merge(h1, on=["MasterFundCode", "Year"], how="left")
df_fee["TotalOperatingCost"] = np.where(
    df_fee["Half"] == 2,
    df_fee["TotalOperatingCost"] - df_fee["TotalOperatingCost_H1"],
    df_fee["TotalOperatingCost"]
)
df_fee.drop(columns=["TotalOperatingCost_H1"], inplace=True)
df_fee = df_fee[['MasterFundCode','Year','Half','TotalOperatingCost']].copy()
# -------- 映射表：确保 Symbol -> MasterFundCode 一对一 --------
df_inst_m = df_inst.merge(map_s2m, on='Symbol', how='left')
df_inst_m['EndDate'] = pd.to_datetime(df_inst_m['EndDate'], errors='coerce')
df_inst_m['InstitutionRatio'] = pd.to_numeric(df_inst_m['InstitutionRatio'], errors='coerce')

df_inst_m['Year'] = df_inst_m['EndDate'].dt.year
df_inst_m['Half'] = df_inst_m['ReportTypeID'].map({5: 1, 6: 2}).astype('Int64')

df_inst_m = df_inst_m.dropna(subset=['MasterFundCode', 'Year', 'Half', 'InstitutionRatio']).copy()

# 关键：同一 MasterFundCode-Year-Half 下（多份额/多行）取最大值，得到唯一键
df_inst = (
    df_inst_m.groupby(['MasterFundCode', 'Year', 'Half'], as_index=False)['InstitutionRatio']
             .max()
)
df_inst['Period'] = df_inst['Year'] * 2 + (df_inst['Half'] - 1)

# 2) 按基金和时间排序
df_inst = df_inst.sort_values(['MasterFundCode', 'Period'])

# 3) 一期（半年）滞后
df_inst['InstitutionRatio_L1'] = df_inst.groupby('MasterFundCode')['InstitutionRatio'].shift(1)

# 可选：清理辅助列
df_inst = df_inst.drop(columns=['Period'])

In [37]:
df_all = df_ta.merge(df_fee , on = ['MasterFundCode', 'Year', 'Half'] ,how = 'inner')
df_all = df_all.merge(df_inst , on = ['MasterFundCode', 'Year', 'Half'] ,how = 'inner')
df_all = df_all.merge(df_turn_std , on = ['MasterFundCode', 'Year', 'Half'] ,how = 'inner')
df_all['Expense_i,t'] = df_all['TotalOperatingCost']/df_all['TotalAsset']
df_all['LnTNA_i,t'] = np.log(df_all['TotalAsset'])
df_all = df_alpha_half.merge(df_all , on =['MasterFundCode', 'Year', 'Half'], how = 'inner')
df_style = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\funds info\FUND_MainInfo.csv")
df_style = df_style[['MasterFundCode','InvestmentStyle']]
df_style = df_style.drop_duplicates(subset=['MasterFundCode'], keep='last').copy()
df_all = df_all.merge(df_style, on =['MasterFundCode'], how = 'left')
df_ms = pd.read_csv(r"D:\studystudy\Window Dressing\raw data\independent variable\others\ManagerSkill.csv")
df_all = df_all.merge(df_ms , on =['MasterFundCode', 'Year', 'Half'], how = 'inner')

In [38]:
df_IDV = df_all[['MasterFundCode','Year','Half','InstitutionRatio_L1','Alpha_6m'
                ,'Expense_i,t','TurningRate','LnTNA_i,t','ManagerSkill_t_1','InvestmentStyle']]
df_IDV.rename(columns={'InstitutionRatio_L1': 'Inst_i,t-1','Alpha_6m': 'Alpha_i,t',
                       'TurningRate': 'Turnover_i,t','ManagerSkill_t_1': 'ManagerSkill_i,t-1'}, inplace=True)

C:\Users\MLTZ\AppData\Local\Temp\ipykernel_22492\3404124051.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_IDV.rename(columns={'InstitutionRatio_L1': 'Inst_i,t-1','Alpha_6m': 'Alpha_i,t',


In [39]:
df_IDV.to_csv(r'D:\studystudy\Window Dressing\merged data\merged_IndependentVariable.csv', index=False, sep=',', encoding='utf-8-sig')

In [40]:
df_DV = pd.read_csv(r"D:\studystudy\Window Dressing\merged data\merged_DependentVariable.csv")

In [41]:
df_reg = df_IDV.merge(df_DV, on =['MasterFundCode', 'Year', 'Half'], how = 'inner')
df_reg = df_reg.replace([np.inf, -np.inf], np.nan).dropna()

In [42]:
df_reg.to_csv(r'D:\studystudy\Window Dressing\merged data\reg1.csv', index=False, sep=',', encoding='utf-8-sig')

In [43]:
X_cols = ['BHRG','RankGap','resBHRG','Inst_i,t-1','Alpha_i,t','Expense_i,t','Turnover_i,t','LnTNA_i,t','ManagerSkill_i,t-1']
df_w = df_reg.copy()
q01 = df_w[X_cols].quantile(0.01)
q99 = df_w[X_cols].quantile(0.99)
df_w[X_cols] = df_w[X_cols].clip(q01, q99, axis=1)
df_reg = df_w.copy()

In [44]:
df_reg_2014 = df_reg.copy()
# 半年度序号：2010H1 -> 2010*2+0, 2010H2 -> 2010*2+1
df_reg_2014['Period'] = df_reg_2014['Year'] * 2 + (df_reg_2014['Half'] - 1)
# 区间：2010H1(6/30) 到 2014H1(6/30)
start_p = 2010 * 2 + (1 - 1)  # 2010H1
end_p   = 2014 * 2 + (1 - 1)  # 2014H1
df_reg_2014 = df_reg_2014[(df_reg_2014['Period'] >= start_p) & (df_reg_2014['Period'] <= end_p)].copy()
# 基金在该区间内至少 4 条记录
mask = df_reg_2014.groupby('MasterFundCode')['Period'].transform('size') >= 4
df_reg_2014 = df_reg_2014[mask].drop(columns='Period').copy()

In [45]:
from linearmodels.iv import AbsorbingLS
for i in ['BHRG','resBHRG', 'RankGap']:
    # 1) 准备 y, X
    y = df_reg_2014[i]
    X = df_reg_2014[['Inst_i,t-1', 'Alpha_i,t', 'Expense_i,t', 'Turnover_i,t', 'LnTNA_i,t', 'ManagerSkill_i,t-1']]
    # 2) 固定效应（Style + Year）
    effects = df_reg_2014[['InvestmentStyle', 'Year']].astype('category')
    # 3) 拼成回归用的数据，并去掉缺失（保证 y/X/effects 同步）
    reg_df = pd.concat([y, X, effects], axis=1).dropna()
    y_clean = reg_df[i]
    X_clean = reg_df[['Inst_i,t-1', 'Alpha_i,t', 'Expense_i,t', 'Turnover_i,t', 'LnTNA_i,t', 'ManagerSkill_i,t-1']]
    effects_clean = reg_df[['InvestmentStyle', 'Year']].astype('category')
    # 4) 估计：AbsorbingLS 会自动包含截距（等价于有 constant）
    mod = AbsorbingLS(y_clean, X_clean, absorb=effects_clean)
    # 5) 标准误：先用稳健（HC1 类似）
    res = mod.fit(
        cov_type='clustered',
        clusters=df_reg.loc[y_clean.index, 'MasterFundCode'])
    print(res.summary)
    print('\n'*3)

                         Absorbing LS Estimation Summary                          
Dep. Variable:                   BHRG   R-squared:                          0.3078
Estimator:               Absorbing LS   Adj. R-squared:                     0.3027
No. Observations:                2457   F-statistic:                        371.06
Date:                Wed, Jan 21 2026   P-value (F-stat):                   0.0000
Time:                        22:36:22   Distribution:                      chi2(6)
Cov. Estimator:             clustered   R-squared (No Effects):             0.1722
                                        Variables Absorbed:                 13.000
                                 Parameter Estimates                                  
                    Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------------
Inst_i,t-1            -0.0002     0.0002    -0.9055     0.3652     -0.0006 

In [47]:
from linearmodels.iv import AbsorbingLS
for i in ['BHRG','resBHRG', 'RankGap']:
    # 1) 准备 y, X
    y = df_reg[i]
    X = df_reg[['Inst_i,t-1', 'Alpha_i,t', 'Expense_i,t', 'Turnover_i,t', 'LnTNA_i,t', 'ManagerSkill_i,t-1']]
    # 2) 固定效应（Style + Year）
    effects = df_reg[['InvestmentStyle', 'Year']].astype('category')
    # 3) 拼成回归用的数据，并去掉缺失（保证 y/X/effects 同步）
    reg_df = pd.concat([y, X, effects], axis=1).dropna()
    y_clean = reg_df[i]
    X_clean = reg_df[['Inst_i,t-1', 'Alpha_i,t', 'Expense_i,t', 'Turnover_i,t', 'LnTNA_i,t', 'ManagerSkill_i,t-1']]
    effects_clean = reg_df[['InvestmentStyle', 'Year']].astype('category')
    # 4) 估计：AbsorbingLS 会自动包含截距（等价于有 constant）
    mod = AbsorbingLS(y_clean, X_clean, absorb=effects_clean)
    # 5) 标准误：先用稳健（HC1 类似）
    res = mod.fit(
        cov_type='clustered',
        clusters=df_reg.loc[y_clean.index, 'MasterFundCode'])
    print(res.summary)
    print('\n'*3)


                         Absorbing LS Estimation Summary                          
Dep. Variable:                   BHRG   R-squared:                          0.1679
Estimator:               Absorbing LS   Adj. R-squared:                     0.1673
No. Observations:               42104   F-statistic:                        1787.0
Date:                Wed, Jan 21 2026   P-value (F-stat):                   0.0000
Time:                        22:42:40   Distribution:                      chi2(6)
Cov. Estimator:             clustered   R-squared (No Effects):             0.0897
                                        Variables Absorbed:                 27.000
                                 Parameter Estimates                                  
                    Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------------
Inst_i,t-1            -0.0005  4.523e-05    -11.396     0.0000     -0.0006 

In [48]:
cols = ['BHRG','RankGap','resBHRG','Inst_i,t-1','Alpha_i,t','Expense_i,t','Turnover_i,t','LnTNA_i,t','ManagerSkill_i,t-1']
desc = df_reg[cols].agg(['count','mean','std','min','median','max']).T
desc['p25'] = df_reg[cols].quantile(0.25)
desc['p75'] = df_reg[cols].quantile(0.75)
desc = desc[['count','mean','std','min','p25','median','p75','max']]
desc

,count,mean,std,min,p25,median,p75,max
BHRG,42104.0,0.161876,0.260074,-0.443947,-0.001822,0.121610,0.276219,1.243200
RankGap,42104.0,-0.016527,0.389698,-0.847906,-0.292693,-0.022089,0.256155,0.843014
resBHRG,42104.0,0.193520,0.252277,-0.523037,0.050744,0.166790,0.305346,1.214053
"Inst_i,t-1",42104.0,33.069711,36.597109,0.000000,0.900000,15.870000,64.022500,100.000000
"Alpha_i,t",42104.0,0.000532,0.066574,-0.190697,-0.033262,-0.000747,0.033646,0.193891
"Expense_i,t",42104.0,0.011622,0.006081,0.003232,0.007935,0.010109,0.013634,0.040509
"Turnover_i,t",42104.0,2.155939,1.915738,0.120991,0.876952,1.583884,2.761144,10.548353
"LnTNA_i,t",42104.0,19.878534,1.595159,16.037482,18.726542,19.947654,21.017947,23.284371
"ManagerSkill_i,t-1",42104.0,0.013487,0.178756,-0.619689,-0.067416,0.025319,0.104164,0.666320
